# Getting started with Megamicros objects

The `Megamicros` class is the base class for all *Megamicros* usb antennas. 
This Notebook shows how you can use the `Megamicros` interface provided you have a *Megamicros* device connected on your USB port.

Beware that you need to have `libusb` installed, both the python interface and the C libusb.1.0.26 library

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from megamicros_tools.log import log
from megamicros.core.mu import Megamicros

log.setLevel( "INFO" )

## Check Usb device and Megamicros device

In [ ]:
Megamicros.check_device()
Megamicros.selfest()

## Declare your antenna object

The ``Megamicros`` constructor needs no argument. The device is detected and the class device parameters are populated.

In [ ]:
# Define an empty antenna
antenna = Megamicros()

Lets know some properties of the connected antenna:

In [ ]:
print( f"Available MEMs = {antenna.available_mems}" )
print( f"Available analogs = {antenna.available_analogs}" )

Try 1s of acquisition on all available MEMs

In [ ]:
antenna.run( 
    duration=1, 
    mems = [0, 1],
    counter=False, 
    counter_skip=False, 
    status=False,
    datatype = 'int32',
    sampling_frequency=50000
)

# Wait until the thread terminates
antenna.wait()

One can force some MEMs to be available while they actually are not. 
Corresponding channels will stay at 0.

In [ ]:
antenna.setAvailableMems( [i for i in range(32)] )

antenna.run( 
    duration=1, 
    mems = [0, 10],
    counter=False, 
    counter_skip=False, 
    status=False,
    datatype = 'int32',
    sampling_frequency=50000
)

antenna.wait()


Getting data can be done by iterate through the `antenna` object provided the antenna had already recived all the data:

In [ ]:
# get data
for data in antenna:
    print( f"data={data}" )

The previous method suggests that real time is not possible.
Actually, you can perform real time acqusition if you iterate before calling the `wait` method, just after the `run` call:   

In [ ]:
import time

antenna.run( 
    duration=1, 
    mems = [0, 1],
    counter=False, 
    counter_skip=False, 
    status=False,
    sampling_frequency=50000
)

# get data
start_time = time.time()
for data in antenna:
    print( f"data={data}" )

print( f"Elapsed time= {time.time() - start_time}" )

# Wait until the thread terminates
antenna.wait()

Note that the elapsed time reaches 4 seconds, whereas the requested acquisition time was 1 second.
Moreover, the system message reported 0.94s.

This is because the system's time measurement method is more accurate: it only includes the actual acquisition time.

The user method, as reported above, includes, in addition to the acquisition:
* The MEMs initialization delay (MEMs powering), default is 1 second
* The timeout delay to leave the queue (2 seconds)
As a consequence, the total elapsed is 4 secondes.

## Plots

In [2]:
import pyqtgraph as pg
from pyqtgraph.Qt import QtCore
from megamicros_tools.log import log
from megamicros.core.mu import Megamicros

# Define an empty antenna
antenna = Megamicros()

# Set the 32 available MEMs
MEMS = [i for i in range(32)]
antenna.setAvailableMems( MEMS )

def plot_on_the_fly( antenna: Megamicros, curves ):
    """ Get last queued signal and plot it

    Parameters
    ----------
    antenna: MemsArrayWS
        The remote antenna
    curves: 
        PyQtGraph curves
    """

    try:
        data = antenna.queue.get( timeout=1 )
    except antenna.queue.Empty:
        return

    t = np.arange( np.size( data, 1 ) )/antenna.sampling_frequency
    for s in range( antenna.mems_number ):
        curves[s].setData( t, ( data[s,:] * antenna.sensibility ) + s - antenna.mems_number/2 )

# init PyQtgraph
win = pg.GraphicsLayoutWidget(show=True, title="Plotting database signals")
win.resize(1000,600)
win.setWindowTitle('Plotting database signals')
pg.setConfigOptions(antialias=True)
graph = win.addPlot(title="Microphones")
graph.setYRange(-5,5, padding=0, update = False)
curves = []
for s in range( len( MEMS ) ):
    curves.append( graph.plot(pen='y' ) )

# Set the Qt timer
timer = QtCore.QTimer()
timer.timeout.connect( lambda: plot_on_the_fly( antenna, curves ) )

In [3]:
antenna.run( 
    mems=MEMS,									# activated mems
    duration=1,
    sampling_frequency=50000,					# sampling frequency
    counter = False,
    counter_skip=False, 
    status=False,
)

# Start and set the timer period in milliseconds
timer.start( 1 )

input( "Press [Return] key to stop...\n" )

timer.stop()
antenna.wait()

: 